# NLTK + SentencePiece + RNN NLP 프로젝트

아래 노트북은 과제 순서에 맞춰 NLTK `movie_reviews` 데이터를 분리하고, 학습 데이터로 SentencePiece 단어사전을 학습한 뒤, PyTorch RNN 분류 모델을 학습/평가합니다.

분류 문제는 영화 리뷰의 `pos`/`neg` 이진 감성 분류입니다.

## 0. 라이브러리와 입력 파라미터

In [1]:
from pathlib import Path
import copy
import random
import nltk
import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# 입력이 필요한 주요 파라미터
VOCAB_SIZE = 4000          # SentencePiece vocabulary 크기
MAX_LEN = 384              # 문장 최대 길이
EMBEDDING_DIM = 128        # embedding vector 차원
HIDDEN_DIM = 128           # RNN hidden state 크기
BATCH_SIZE = 32
EPOCHS = 8
LEARNING_RATE = 5e-4
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 2

# 출력층 구조 및 최종 아웃풋
NUM_CLASSES = 2            # neg, pos
# 모델의 최종 출력은 [batch_size, 2] 크기의 class logits입니다.
# 학습 시 CrossEntropyLoss가 logits에 softmax를 내부적으로 적용합니다.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cpu')

## 1. NLTK 데이터를 학습/테스트 데이터로 9:1 분리

In [2]:
try:
    nltk.data.find('corpora/movie_reviews')
except LookupError:
    nltk.download('movie_reviews')

from nltk.corpus import movie_reviews

label_to_id = {'neg': 0, 'pos': 1}

documents = []
for category in movie_reviews.categories():
    for file_id in movie_reviews.fileids(category):
        text = ' '.join(movie_reviews.words(file_id))
        documents.append((text, label_to_id[category]))

random.shuffle(documents)

test_size = int(len(documents) * 0.1)
test_data = documents[:test_size]
train_full_data = documents[test_size:]

print(f'total: {len(documents)}')
print(f'train_full: {len(train_full_data)}')
print(f'test: {len(test_data)}')

total: 2000
train_full: 1800
test: 200


## 2. 학습 데이터로 SentencePiece 단어사전 생성 및 토큰화

In [3]:
work_dir = Path('sentencepiece_artifacts')
work_dir.mkdir(exist_ok=True)

corpus_path = work_dir / 'movie_reviews_train_corpus.txt'
model_prefix = work_dir / 'movie_reviews_spm'

with corpus_path.open('w', encoding='utf-8') as f:
    for text, _ in train_full_data:
        f.write(text.replace('\n', ' ') + '\n')

spm.SentencePieceTrainer.train(
    input=str(corpus_path),
    model_prefix=str(model_prefix),
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    character_coverage=1.0,
    max_sentence_length=10000,
)

sp = spm.SentencePieceProcessor(model_file=str(model_prefix) + '.model')
PAD_ID = sp.pad_id()
UNK_ID = sp.unk_id()

sample_ids = sp.encode(train_full_data[0][0][:200], out_type=int)

print('vocab size:', sp.get_piece_size())
print('pad id:', PAD_ID, 'unk id:', UNK_ID)
print('sample token ids:', sample_ids[:30])

vocab size: 4000
pad id: 0 unk id: 1
sample token ids: [364, 567, 3946, 50, 262, 3360, 46, 5, 1395, 3969, 3938, 65, 46, 3941, 38, 484, 27, 327, 116, 839, 2008, 563, 134, 182, 973, 1060, 32, 249, 187, 15]


## 3. 학습 데이터를 Train/Validation 데이터로 8:2 분리

In [4]:
val_size = int(len(train_full_data) * 0.2)
val_data = train_full_data[:val_size]
train_data = train_full_data[val_size:]

print(f'train: {len(train_data)}')
print(f'validation: {len(val_data)}')
print(f'test: {len(test_data)}')

train: 1440
validation: 360
test: 200


## 4. 단어사전으로 Train/Validation/Test 토큰화, Padding, Torch Dataset 생성

In [5]:
def encode_and_pad(text, max_len=MAX_LEN):
    ids = sp.encode(text, out_type=int)
    ids = ids[:max_len]
    attention_length = len(ids)
    if len(ids) < max_len:
        ids = ids + [PAD_ID] * (max_len - len(ids))
    return ids, attention_length


def build_dataset(data):
    input_ids = []
    lengths = []
    labels = []
    for text, label in data:
        ids, length = encode_and_pad(text)
        input_ids.append(ids)
        lengths.append(length)
        labels.append(label)

    return TensorDataset(
        torch.tensor(input_ids, dtype=torch.long),
        torch.tensor(lengths, dtype=torch.long),
        torch.tensor(labels, dtype=torch.long),
    )


train_dataset = build_dataset(train_data)
val_dataset = build_dataset(val_data)
test_dataset = build_dataset(test_data)

print(train_dataset.tensors[0].shape, train_dataset.tensors[1].shape, train_dataset.tensors[2].shape)
print(val_dataset.tensors[0].shape, test_dataset.tensors[0].shape)

torch.Size([1440, 384]) torch.Size([1440]) torch.Size([1440])
torch.Size([360, 384]) torch.Size([200, 384])


## 5. DataLoader 생성

In [6]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch_input_ids, batch_lengths, batch_labels = next(iter(train_loader))
print(batch_input_ids.shape, batch_lengths.shape, batch_labels.shape)

torch.Size([32, 384]) torch.Size([32]) torch.Size([32])


## 6. Embedding Layer - BiLSTM - 출력층 모델 생성

In [7]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_id, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.embedding_dropout = nn.Dropout(dropout)
        self.rnn = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.output_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, lengths):
        embedded = self.embedding_dropout(self.embedding(input_ids))
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.rnn(packed)
        last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        logits = self.fc(self.output_dropout(last_hidden))
        return logits


model = RNNClassifier(
    vocab_size=sp.get_piece_size(),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    pad_id=PAD_ID,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

model

RNNClassifier(
  (embedding): Embedding(4000, 128, padding_idx=0)
  (embedding_dropout): Dropout(p=0.3, inplace=False)
  (rnn): LSTM(128, 128, batch_first=True, bidirectional=True)
  (output_dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=2, bias=True)
)

## 7. 모델 학습

In [8]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for input_ids, lengths, labels in loader:
            input_ids = input_ids.to(DEVICE)
            lengths = lengths.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += batch_size

    return total_loss / total, correct / total


history = []
best_val_loss = float('inf')
best_state = None
patience_count = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
    })
    print(
        f'Epoch {epoch:02d} | '
        f'train loss {train_loss:.4f}, acc {train_acc:.4f} | '
        f'val loss {val_loss:.4f}, acc {val_acc:.4f}'
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= EARLY_STOPPING_PATIENCE:
            print(f'Early stopping at epoch {epoch:02d}')
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f'Restored best model with validation loss {best_val_loss:.4f}')

Epoch 01 | train loss 0.6962, acc 0.4924 | val loss 0.6996, acc 0.4806
Epoch 02 | train loss 0.6785, acc 0.5833 | val loss 0.7029, acc 0.4639
Epoch 03 | train loss 0.6616, acc 0.6250 | val loss 0.7004, acc 0.4917
Early stopping at epoch 03
Restored best model with validation loss 0.6996


## 8. 모델 평가

In [9]:
from sklearn.metrics import classification_report, confusion_matrix


def collect_predictions(model, loader):
    model.eval()
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for input_ids, lengths, labels in loader:
            input_ids = input_ids.to(DEVICE)
            lengths = lengths.to(DEVICE)
            logits = model(input_ids, lengths)
            predictions = logits.argmax(dim=1).cpu()

            all_labels.extend(labels.tolist())
            all_predictions.extend(predictions.tolist())

    return all_labels, all_predictions


test_loss, test_acc = run_epoch(model, test_loader, criterion)
test_labels, test_predictions = collect_predictions(model, test_loader)

print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')
print('\nClassification report')
print(classification_report(test_labels, test_predictions, target_names=['neg', 'pos']))
print('Confusion matrix')
print(confusion_matrix(test_labels, test_predictions))

Test loss: 0.6861
Test accuracy: 0.5450

Classification report
              precision    recall  f1-score   support

         neg       0.48      0.72      0.58        87
         pos       0.66      0.41      0.50       113

    accuracy                           0.55       200
   macro avg       0.57      0.57      0.54       200
weighted avg       0.58      0.55      0.54       200

Confusion matrix
[[63 24]
 [67 46]]


In [10]:
def predict_sentiment(text):
    model.eval()
    ids, length = encode_and_pad(text)
    input_ids = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    lengths = torch.tensor([length], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids, lengths)
        probs = torch.softmax(logits, dim=1).squeeze(0)
        pred_id = int(probs.argmax().item())

    id_to_label = {0: 'neg', 1: 'pos'}
    return id_to_label[pred_id], probs.cpu().tolist()


sample_text = 'This movie was surprisingly fun and emotionally satisfying.'
predict_sentiment(sample_text)

('pos', [0.4738280475139618, 0.526171863079071])